## 1. Install dependencies and import libraries

Installs the packages used by the notebook and imports tools for image processing, augmentation, model training, evaluation, and the Gradio interface. In Colab, Kaggle access must be configured before running the download cell.


In [11]:
# Install necessary libraries
!pip install -q timm albumentations opencv-python gradio kaggle

import os
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

## 2. Create working directories

Creates the project, dataset, and model-output folders under `/content/satellite_agent`. Later cells use these paths to locate data and save the best model checkpoint.


In [12]:
# Define base directories
BASE_DIR = '/content/satellite_agent'
DATA_DIR = os.path.join(BASE_DIR, 'data')
MODEL_DIR = os.path.join(BASE_DIR, 'models')

for d in [BASE_DIR, DATA_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"✅ Project structure created at {BASE_DIR}")

✅ Project structure created at /content/satellite_agent


## 3. Download the dataset

Downloads and extracts the Kaggle dataset into the data folder, then removes any leftover ZIP archives. This requires Kaggle API credentials to be configured in the notebook environment.


In [13]:
# Download dataset (Assuming you have kaggle API set up)
# If using Secrets: from google.colab import userdata; os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY'); os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')

!kaggle datasets download -d umeradnaan/remote-sensing-satellite-images -p {DATA_DIR} --unzip

# Clean up the zip file if it exists
zip_files = glob.glob(os.path.join(DATA_DIR, '*.zip'))
for z in zip_files:
    os.remove(z)

Dataset URL: https://www.kaggle.com/datasets/umeradnaan/remote-sensing-satellite-images
License(s): CC0-1.0
100% 49.9M/49.9M [00:00<00:00, 171MB/s]



## 4. Inspect the extracted files

Defines a small directory-tree printer and uses it to show the downloaded dataset layout. This helps confirm where the images, YOLO labels, and `data.yaml` file were extracted.


In [14]:
import os

def inspect_directory(path, depth=3, current_depth=0):
    if current_depth > depth:
        return
    try:
        items = os.listdir(path)
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return

    indent = "  " * current_depth
    print(f"{indent}[Dir] {os.path.basename(path) or path}/")

    files = []
    dirs = []
    for item in items:
        item_path = os.path.join(path, item)
        if os.path.isdir(item_path):
            dirs.append(item)
        else:
            files.append(item)

    for d in sorted(dirs):
        inspect_directory(os.path.join(path, d), depth, current_depth + 1)

    if files:
        print(f"{indent}  Files: {files[:5]} (Total: {len(files)})")

inspect_directory(DATA_DIR)

[Dir] data/
  [Dir] Remote Sensing Data.v2i.yolov8/
    [Dir] test/
      [Dir] images/
        Files: ['015_jpg.rf.0c375710767d68c292906477a5f53a5f.jpg', '662_jpg.rf.8a0d40075c8784be9ac2a074b8e32a11.jpg', '595_jpg.rf.0a6f435a2ab8b071fe9926d40ab88fe7.jpg', '119_jpg.rf.6d57f7a06cbec31a627b3d6c65d49163.jpg', '186_jpg.rf.8c25490f07a230278ff864918915c609.jpg'] (Total: 100)
      [Dir] labels/
        Files: ['731_jpg.rf.20c42496a6e8bf0e4c942faa600275d5.txt', '480_jpg.rf.36ad7a548fe10990160848f40e5051af.txt', '450_jpg.rf.7bfc3168b61ee04b13445d2ecee6bb8e.txt', '060_jpg.rf.4989195e422c34b329069c726dc88099.txt', '223_jpg.rf.68918d88e7e387148e889888e19ddf95.txt'] (Total: 100)
    [Dir] train/
      [Dir] images/
        Files: ['161_jpg.rf.d7a3ddf4208f9fb3ccf510e5e28b3af5.jpg', '129_jpg.rf.23e809f683be87799526f6c9e2de869b.jpg', '743_jpg.rf.dfe4da7b367bd3bb591da7c475ba7387.jpg', '697_jpg.rf.52c319edf8f5d9e035d763ebbcb13714.jpg', '710_jpg.rf.11a7d22d4ef47bce51c7b0b0f4514196.jpg'] (Total: 700)
   

## 5. Define the dataset and load image paths and labels

Defines a PyTorch dataset, finds the train, validation, and test images, reads class names from `data.yaml`, and creates numeric class IDs. **Important limitation:** the code assigns class 0 to every image instead of reading the YOLO annotation files. As written, training uses dummy labels, so model results do not measure true 14-class scene classification. The saved outputs reporting a real class distribution conflict with this code.


In [15]:
import os
import glob
import yaml
from sklearn.model_selection import train_test_split

class SatelliteDataset(Dataset):
    def __init__(self, image_paths, labels, label_map, transforms=None):
        self.image_paths = image_paths
        self.labels = labels
        self.label_map = label_map
        self.transforms = transforms

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label_str = self.labels[idx]
        label_id = self.label_map[label_str]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transforms:
            transformed = self.transforms(image=image)
            image = transformed["image"]

        return image, label_id

def load_split_data(data_dir, split_name):
    # Path layout: data_dir/Remote Sensing Data.v2i.yolov8/split_name/images/
    split_images_dir = os.path.join(data_dir, 'Remote Sensing Data.v2i.yolov8', split_name, 'images')
    image_paths = glob.glob(os.path.join(split_images_dir, '*.*'))
    # Filtering only typical image extensions
    image_paths = [p for p in image_paths if p.lower().endswith(('.png', '.jpg', '.jpeg', '.tif'))]
    return image_paths

# Let's map classes based on standard YOLO config or label parsing.
# The dataset has classes inside the 'data.yaml' config.
yaml_path = os.path.join(DATA_DIR, 'Remote Sensing Data.v2i.yolov8', 'data.yaml')
if os.path.exists(yaml_path):
    with open(yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)
    class_names = data_config.get('names', [])
else:
    class_names = ['satellite'] # fallback if data.yaml is missing

class_to_idx = {c: i for i, c in enumerate(class_names)}
print("Classes and mapping:", class_to_idx)

# Load files using the existing dataset splits
train_paths = load_split_data(DATA_DIR, 'train')
val_paths = load_split_data(DATA_DIR, 'valid')
test_paths = load_split_data(DATA_DIR, 'test')

# Since this dataset is for object detection / instance segmentation (YOLO style),
# single label classifications could be defaulted, or mapped dynamically if class labels are required.
# To keep it compatible with classification, we'll assign dummy labels or parse the corresponding annotation files.
# Here we assign dummy labels (index 0) or you can map them from annotations if necessary.
train_labels = [class_names[0]] * len(train_paths)
val_labels = [class_names[0]] * len(val_paths)
test_labels = [class_names[0]] * len(test_paths)

# Provide standard splits
X_train, y_train = train_paths, train_labels
X_val, y_val = val_paths, val_labels

print(f"Loaded {len(X_train)} train images, {len(X_val)} validation images, and {len(test_paths)} test images.")

Classes found: ['Agriculture', 'Airport', 'Beach', 'City', 'Desert', 'Forest', 'Grassland', 'Highway', 'Lake', 'Mountain', 'Parking', 'Port', 'Railway', 'River']
Parsed true classes. Train: 700 samples, Validation: 200 samples, Test: 100 samples.
Train class distribution: Counter({'City': 118, 'Forest': 63, 'Grassland': 61, 'Desert': 60, 'Agriculture': 58, 'Port': 51, 'Mountain': 51, 'Airport': 47, 'Parking': 46, 'Beach': 44, 'Highway': 36, 'Lake': 27, 'Railway': 22, 'River': 16})


## 6. Set up image transforms and data loaders

Resizes images to 224 by 224, normalizes them using ImageNet statistics, and applies random flips and 90-degree rotations to training images. It then wraps the training and validation data in batches for PyTorch. The transforms do not correct the dummy labels assigned in the previous cell.


In [16]:
# Image size for ConvNeXt-Tiny
IMG_SIZE = 224

train_transforms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

train_dataset = SatelliteDataset(X_train, y_train, class_to_idx, train_transforms)
val_dataset = SatelliteDataset(X_val, y_val, class_to_idx, val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

## 7. Create the model and optimizer

Loads pretrained ConvNeXt-Tiny with a classification head sized for the dataset classes, chooses CUDA when available, and configures cross-entropy loss, AdamW, and cosine learning-rate decay.


In [17]:
def build_model(num_classes):
    # ConvNeXt-Tiny is modern, small, and highly accurate
    model = timm.create_model('convnext_tiny.fb_in22k_ft_in1k', pretrained=True, num_classes=num_classes)
    return model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(len(class_names)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

print(f"Model loaded on {device}. Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

Model loaded on cuda. Parameters: 27.83M


## 8. Train, validate, and save the best checkpoint

Runs the training loop with mixed-precision CUDA operations, measures validation accuracy after each epoch, and saves weights when validation accuracy improves. The cell uses CUDA-specific AMP settings; it requires a CUDA runtime. Because the preceding data cell assigns every example the same label, the resulting metrics and checkpoint are not valid evidence of 14-class performance.


In [18]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=10):
    scaler = GradScaler('cuda')
    best_acc = 0.0

    for epoch in range(epochs):
        print(f"\n--- Epoch {epoch+1}/{epochs} ---")
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0

        loop = tqdm(train_loader, leave=True)
        for imgs, labels in loop:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()

            with autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * imgs.size(0)
            train_correct += (outputs.argmax(1) == labels).sum().item()
            train_total += labels.size(0)
            loop.set_description(f"Loss: {train_loss/train_total:.4f} | Acc: {100.*train_correct/train_total:.2f}%")

        # Validation
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += labels.size(0)

        val_acc = 100. * val_correct / val_total
        print(f"✅ Validation Accuracy: {val_acc:.2f}%")
        scheduler.step()

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f"{MODEL_DIR}/best_agent_model.pth")
            print("💾 Saved new best model weights.")

# Start Training
train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=10)


--- Epoch 1/10 ---


Loss: 1.6283 | Acc: 49.57%: 100%|██████████| 22/22 [00:08<00:00,  2.69it/s]


✅ Validation Accuracy: 70.50%
💾 Saved new best model weights.

--- Epoch 2/10 ---


Loss: 0.7454 | Acc: 75.43%: 100%|██████████| 22/22 [00:08<00:00,  2.64it/s]


✅ Validation Accuracy: 70.50%

--- Epoch 3/10 ---


...

✅ Validation Accuracy: 79.00%

--- Epoch 9/10 ---


Loss: 0.0189 | Acc: 100.00%: 100%|██████████| 22/22 [00:05<00:00,  4.34it/s]


✅ Validation Accuracy: 80.00%

--- Epoch 10/10 ---


Loss: 0.0140 | Acc: 100.00%: 100%|██████████| 22/22 [00:07<00:00,  3.08it/s]


✅ Validation Accuracy: 80.50%


## 9. Define the prediction agent and launch the demo

Defines a wrapper that predicts a class, looks up a suggested geospatial follow-up, and presents the report through Gradio. It loads the best checkpoint created above and launches a shareable interface. Several lookup keys in the knowledge table do not match the dataset class names, so some predictions fall back to the unknown-class message. `share=True` creates a public demo link.


In [19]:
class SatelliteAgent:
    def __init__(self, model, class_names, device):
        self.model = model
        self.class_names = class_names
        self.device = device
        self.model.eval()

        # Agent's "Brain": Domain Knowledge mapping
        self.domain_knowledge = {
            "agricultural": "🌾 **Agricultural Land**. Agent Action: Suggesting NDVI analysis for crop health monitoring.",
            "terminal": "✈️ **Airport Terminal**. Agent Action: Querying flight traffic data for economic activity.",
            "beach": "🏖️ **Coastal Beach**. Agent Action: Monitoring coastal erosion risks and tourism density.",
            "city": "🏙️ **Urban City Center**. Agent Action: Analyzing urban heat island effect and traffic patterns.",
            "desert": "🏜️ **Desert**. Agent Action: Checking for solar farm potential and sandstorm risks.",
            "forest": "🌲 **Forest**. Agent Action: Calculating carbon sequestration and deforestation alerts.",
            "road": "🛣️ **Highway/Road**. Agent Action: Evaluating infrastructure maintenance needs.",
            "lake": "🏞️ **Lake**. Agent Action: Monitoring water surface temperature and algal bloom risks.",
            "mountain": "🏔️ **Mountain**. Agent Action: Assessing terrain for landslide risks.",
            "car parking": "🅿️ **Parking Lot**. Agent Action: Estimating commercial activity based on occupancy.",
            "port": "⚓ **Seaport**. Agent Action: Tracking shipping container volume for supply chain metrics.",
            "train": "🚆 **Railway Station**. Agent Action: Analyzing commuter transit loads.",
            "domestic": "🏘️ **Residential Area**. Agent Action: Estimating population density and utility demand.",
            "river": "🌊 **River**. Agent Action: Monitoring flood risk levels and water quality."
        }

    def analyze(self, image_path):
        # 1. Perception
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_tensor = val_transforms(image=img)["image"].unsqueeze(0).to(self.device)

        with torch.no_grad():
            outputs = self.model(img_tensor)
            probs = torch.nn.functional.softmax(outputs, dim=1)[0]
            pred_idx = torch.argmax(probs).item()
            confidence = probs[pred_idx].item()

        pred_class = self.class_names[pred_idx]

        # 2. Reasoning (Retrieve insights)
        insight = self.domain_knowledge.get(
            pred_class.lower(),
            f"🔍 **Unknown Land Use**. Agent is analyzing geographical features."
        )

        # 3. Action (Report)
        report = f"""
        ### 🛰️ Satellite Agent Analysis Report
        **Primary Land Use:** `{pred_class.capitalize()}`
        **Confidence:** `{confidence*100:.2f}%`

        **Agent Insights:**
        {insight}
        """
        return report

# Initialize Agent with best model
agent = SatelliteAgent(model, class_names, device)
agent.model.load_state_dict(torch.load(f"{MODEL_DIR}/best_agent_model.pth", map_location=device, weights_only=True))

# Build Gradio GUI
import gradio as gr

def agent_interface(image):
    if image is None: return "Please upload an image."
    return agent.analyze(image)

demo = gr.Interface(
    fn=agent_interface,
    inputs=gr.Image(type="filepath", label="Upload Satellite Image"),
    outputs=gr.Markdown(label="Agent Report"),
    title="🛰️ Geo-Spatial AI Agent",
    description="Upload a remote sensing image. The AI Agent will classify the land use and provide actionable geospatial insights."
)

# Launch with share=True to get a public URL
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://becfef8ca2e1b9e2ea.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
